# Experiment 1 — <Strategy_Name> (the benchmark)

**The pure idea, and the yardstick every later experiment is measured against.**

> <The main idea from `OBJECTIVE.md`, in one sentence.>

| Decision | Rule |
| --- | --- |
| **Selection** | <the eligibility condition, naming the column> |
| **Ranking** | <the ranking column, highest first> |
| **Holdings** | <the holding count> |
| **Weighting** | <the scheme — no cap, no floor, or say which> |
| **Rebalancing** | <calendar, or event-driven on a stated trigger> |
| **Residual** | parked in `BIL` (1–3 month T-bills), the tradable stand-in for cash |

Why each pillar is there is argued in `OBJECTIVE.md`. The hypothesis is `BLUEPRINT_1.md`, the
running log `JOURNAL_1.md`, and the results that survive `FINDINGS_1.md`.

## Position in the pipeline

This notebook is **only the strategy**. The universe and the data are built by earlier stages and
are simply read here:

```
Universe/universe.ipynb   ->  Security_Master.csv
Data/curator.py           ->  Data/Curator/Time_Series/     m_* + c_*
Data/refinery.py          ->  Data/Refinery/Time_Series/    + r_*      <- this notebook reads here
Data/analyzer.ipynb       ->  the EDA behind the feature choices
        |
        v
experiment_1.ipynb        ->  Portfolio/  ->  Backtest/  ->  Attribution/
```

## What this notebook does *not* do

It does not download anything, profile the universe, or compute signals. If a number about the
data itself is needed, it belongs in the Universe or Data stage — that separation is what keeps
every experiment comparable, because all of them read the identical panel.

**The template's contract.** Every section below is strategy-agnostic except **section 2, the
rule**, which is the one cell you write. It must produce three objects — `selected_matrix`,
`REBALANCE_DATES`, `target_weights` — and everything downstream runs unchanged.

---

## 0 · Setup

Paths, the strategy's columns, and the panel. Only the columns the strategy consumes are read.

The three price columns do **three different jobs**, and getting them out of step is silent — the
backtest P&L and the attribution would quietly run on different bases:

| Role | Column | Why |
| --- | --- | --- |
| Daily mark | `m_close_dividend_and_split_adjusted` | total-return valuation between rebalances |
| Fill | `c_vwap_dividend_and_split_adjusted` | the price a trade actually gets |
| Commission | `c_vwap` | per-share cents ride on the **unadjusted** share count |

The provider returns `m_vwap` and `m_vwap_dividend_and_split_adjusted` as null, which is exactly why
the Curator reconstructs them as `c_*`. Never point anything at the `m_vwap*` columns.

**This is the benchmark, so the loading steps are written inline** rather than imported from
`Experiments/panel.py`: a baseline that cannot be read top to bottom without chasing an import is a
worse baseline. Later experiments import `panel.py` so their notebooks show only what they change.

In [ ]:
"""Experiment 1 - the benchmark. Reads Data/Refinery/, writes this experiment's folders."""
import pathlib
import sys

import matplotlib.pyplot
import numpy
import pandas


def find_repo_root(start):
    """Walk up from `start` to the directory that holds pyproject.toml and Experiments/."""
    for candidate in (start, *start.parents):
        if (candidate / "pyproject.toml").is_file() and (candidate / "Experiments").is_dir():
            return candidate

    return start


REPO_ROOT = find_repo_root(pathlib.Path.cwd())
sys.path.insert(0, str(REPO_ROOT / "Experiments"))
import engine

EXPERIMENT_DIR = REPO_ROOT / "Experiments" / "Experiment_1"
REFINERY_DIR = REPO_ROOT / "Data" / "Refinery" / "Time_Series"
# The engine reads one market-data directory and needs the cash proxy and the benchmarks in it,
# and those are deliberately absent from the Refinery (they are not part of any cross-section).
MARKET_DATA_DIR = REPO_ROOT / "Data" / "Curator" / "Time_Series"
BENCHMARK_DIR = REPO_ROOT / "Data" / "Curator" / "Benchmarks"
UNIVERSE_PATH = REPO_ROOT / "Universe" / "Investable_Universe.csv"

PORTFOLIO_DIR = EXPERIMENT_DIR / "Portfolio"
BACKTEST_DIR = EXPERIMENT_DIR / "Backtest"
ATTRIBUTION_DIR = EXPERIMENT_DIR / "Attribution"
CHART_DIR = PORTFOLIO_DIR / "Charts"
for directory in (PORTFOLIO_DIR, BACKTEST_DIR, ATTRIBUTION_DIR, CHART_DIR):
    directory.mkdir(parents=True, exist_ok=True)

# --- The strategy's columns ----------------------------------------------------------------
# The only constants in this notebook that belong to the strategy rather than to the process.
# They are declared here, not in Experiments/panel.py, so a signal never becomes every later
# experiment's default without anyone deciding it.
DATE_COLUMN = "m_date"
SIGNAL_COLUMN = "c_<your_signal_column>"        # eligibility: 1.0 when a name may be held
SIZING_COLUMN = "c_daily_traded_value_63d"     # ranking and weighting; ADTV ships with the template
MARK_PRICE_COLUMN = "m_close_dividend_and_split_adjusted"
TRADE_PRICE_COLUMN = "c_vwap_dividend_and_split_adjusted"
COMMISSION_PRICE_COLUMN = "c_vwap"

PANEL_COLUMNS = (
    DATE_COLUMN,
    SIGNAL_COLUMN,
    SIZING_COLUMN,
    MARK_PRICE_COLUMN,
    TRADE_PRICE_COLUMN,
    "sector_current",
    "industry_current",
)
CASH_TICKER = engine.CASH_TICKER

paths = sorted(REFINERY_DIR.glob("*.csv"))
assert paths, f"no refined files in {REFINERY_DIR} - run: uv run python Data/refinery.py"

frames = []
for path in paths:
    frame = pandas.read_csv(path, usecols=list(PANEL_COLUMNS), parse_dates=[DATE_COLUMN])
    frame.insert(0, "ticker", path.stem)
    frames.append(frame)

panel = pandas.concat(frames, ignore_index=True)

print(f"Repo root : {REPO_ROOT}")
print(f"Reading   : {REFINERY_DIR.relative_to(REPO_ROOT)}")
print(f"Panel     : {panel['ticker'].nunique()} tickers x {panel[DATE_COLUMN].nunique()} dates"
      f" = {len(panel):,} rows")
print(f"Window    : {panel[DATE_COLUMN].min().date()} -> {panel[DATE_COLUMN].max().date()}")

---

## 1 · One position per company, then reshape to matrices

The universe is point-in-time, so it contains **ticker changes**: two legs sharing an ISIN, each
carrying only part of the history. Left alone they would be two independent positions and the book
would double-count the company at the changeover.

Positions are therefore keyed by **ISIN**, falling back to the ticker where the universe file has
none. Where two legs overlap on a date, the leg still reporting later wins — that is the surviving
listing.

The long panel then becomes one wide `dates × companies` matrix per input, which is what makes the
whole selection rule in section 2 a handful of vectorised lines instead of a loop over files.

In [ ]:
universe = pandas.read_csv(UNIVERSE_PATH, encoding="utf-8-sig", dtype=str)
universe["ticker"] = universe["ticker"].str.strip()

# Company key: ISIN where the universe file has one, the ticker itself otherwise.
company_by_ticker = {
    row.ticker: (row.isin if isinstance(row.isin, str) and row.isin else row.ticker)
    for row in universe.itertuples()
}
panel["company"] = panel["ticker"].map(company_by_ticker).fillna(panel["ticker"])

tickers_by_company = {}
for ticker, company in company_by_ticker.items():
    tickers_by_company.setdefault(company, []).append(ticker)
multi_leg_companies = {
    company
    for company, ticker_list in tickers_by_company.items()
    if len(ticker_list) > 1
}

# Resolve overlaps: sort so the leg that reports latest sorts last, then keep the last row.
last_date_by_ticker = panel.groupby("ticker")[DATE_COLUMN].max()
panel["leg_rank"] = panel["ticker"].map(last_date_by_ticker)
overlapping_rows = int(
    panel[panel["company"].isin(multi_leg_companies)]
    .duplicated(subset=["company", DATE_COLUMN], keep=False)
    .sum()
)

stitched = (
    panel.sort_values(["company", DATE_COLUMN, "leg_rank"])
    .drop_duplicates(subset=["company", DATE_COLUMN], keep="last")
    .drop(columns="leg_rank")
)

company_metadata = (
    stitched.sort_values(DATE_COLUMN)
    .groupby("company")
    .agg(
        ticker=("ticker", "last"),
        sector=("sector_current", "last"),
        industry=("industry_current", "last"),
    )
)
name_by_isin = dict(zip(universe["isin"], universe["name"]))
company_metadata["name"] = pandas.Series(
    company_metadata.index.map(name_by_isin),
    index=company_metadata.index,
).fillna(company_metadata["ticker"])


def to_matrix(frame, column):
    """A dates x companies matrix of `column`, over the union of all observed trading days."""

    return frame.pivot(index=DATE_COLUMN, columns="company", values=column).sort_index()


signal_matrix = to_matrix(stitched, SIGNAL_COLUMN)
sizing_matrix = to_matrix(stitched, SIZING_COLUMN)
mark_price_matrix = to_matrix(stitched, MARK_PRICE_COLUMN)
trade_price_matrix = to_matrix(stitched, TRADE_PRICE_COLUMN)
ticker_matrix = to_matrix(stitched, "ticker")

TRADING_DAYS = signal_matrix.index

print(f"stitched : {len(panel):,} ticker-rows -> {len(stitched):,} company-rows")
print(f"           {stitched['company'].nunique()} companies"
      f" from {panel['ticker'].nunique()} tickers"
      f" ({len(multi_leg_companies)} multi-leg, {overlapping_rows:,} overlapping rows resolved)")
print(f"matrices : {signal_matrix.shape[0]:,} trading days x {signal_matrix.shape[1]} companies")

---

## 2 · The rule — the one cell you write

Three statements, in order, each a full matrix operation so the whole strategy is defined without a
loop over dates:

1. **Who is eligible.** Signal active *and* tradable — a mark and a fill price both exist.
2. **Which names, and how much.** Rank the eligible names on the sizing column **as of the previous
   close**, so no decision uses a price it could not have seen; drop anything that cannot be traded
   today and re-rank the survivors.
3. **When to trade.** Calendar, or only on days the selected set changes.

**The contract this cell must satisfy** — everything below reads exactly these three objects:

| Object | Type | Meaning |
| --- | --- | --- |
| `selected_matrix` | `dates × companies` boolean | what the book holds on each day |
| `REBALANCE_DATES` | `DatetimeIndex` | the days the book is re-struck |
| `target_weights` | `REBALANCE_DATES × companies` float, rows summing to 1.0 | the book on each of those days |

### The one look-ahead, stated plainly

A name that delists must be sold on the **last day it still has a fill price**, and knowing that day
is its last requires seeing the next one. This is the standard backtest compromise — the alternative,
carrying a position that can never be exited, is a larger distortion — and it is implemented below
by making a name ineligible on its final tradable day, so the set changes, the rebalance fires, and
the position is sold while a price still exists. The eligibility scaffold is process, not strategy,
and is written for you.

In [ ]:
HOLDING_COUNT = 30  # <- your holding count

# --- Process: tradability and the documented look-ahead (do not change) ----------------------
tradable_matrix = mark_price_matrix.notna() & trade_price_matrix.notna()

# The last day a name can still be sold. `shift(-1)` is the documented look-ahead; the final row
# of the sample is excluded because the end of the data is not a delisting.
last_tradable_day = tradable_matrix & ~tradable_matrix.shift(-1, fill_value=False)
last_tradable_day.iloc[-1] = False

eligible_matrix = (signal_matrix == 1.0) & tradable_matrix & ~last_tradable_day

# --- Strategy: write the rule here ----------------------------------------------------------
# Produce `selected_matrix`, `REBALANCE_DATES` and `target_weights` from `eligible_matrix` and
# `sizing_matrix`.  Two invariants every rule must respect, and section 2.1 asserts them:
#   - decide on yesterday's close: rank `sizing_matrix.shift(1)`, never today's value;
#   - implement on today's tradable names: a name that dropped out overnight is replaced, not held.
#
# The three assignments below are the contract with its shapes, holding an EMPTY book so the
# notebook lints as a whole; the `raise` stops execution here until a real rule replaces them.
selected_matrix = eligible_matrix & False                 # dates x companies, boolean
REBALANCE_DATES = eligible_matrix.index[:0]               # DatetimeIndex of re-strike days
target_weights = eligible_matrix.iloc[:0].astype(float)   # REBALANCE_DATES x companies, float
msg = (
    "Experiment 1's rule has not been written. Replace the three placeholder assignments in this "
    "cell with the rule, then delete this raise - see the contract in the cell above."
)
raise NotImplementedError(msg)

### 2.1 · Invariants

Cheap to check here, expensive to discover inside a P&L. **Every rule must pass these unchanged.**

In [ ]:
TOLERANCE = 1e-9

row_sums = target_weights.sum(axis=1)
assert numpy.allclose(row_sums, 1.0, atol=1e-8), (
    f"weights must sum to 1.0 on every rebalance; worst = {row_sums.min():.9f}"
)
assert (target_weights >= -TOLERANCE).all().all(), "no negative weights: the strategy is long-only"

held = target_weights > 0
assert (held.sum(axis=1) <= HOLDING_COUNT).all(), "more names held than the rule allows"

# No look-ahead: every name paid for today was eligible at *yesterday's* close.
eligible_yesterday = eligible_matrix.shift(1, fill_value=False).loc[REBALANCE_DATES]
assert not (held & ~eligible_yesterday).any().any(), (
    "a weight was assigned to a name that was not eligible at the prior close"
)

# Every name bought is tradable on the day it is bought, so the fill price exists.
assert not (held & ~tradable_matrix.loc[REBALANCE_DATES]).any().any(), (
    "a weight was assigned to a name with no fill price on its implementation date"
)

# Nothing is still held on a day after it stopped being tradable.
carried = held.astype("boolean").reindex(TRADING_DAYS).ffill().fillna(False).astype(bool)
stranded = carried & ~tradable_matrix
stranded_names = stranded.any(axis=0)
assert not stranded_names.any(), (
    f"{int(stranded_names.sum())} position(s) carried past their last tradable day"
)

print("All invariants hold:")
print("  weights sum to 1.0 on every rebalance date")
print(f"  no negative weights, never more than {HOLDING_COUNT} names")
print("  every holding was eligible at the prior close and tradable when bought")
print("  no position is carried past its last tradable day")

---

## 3 · Construction — is this a book you would actually run?

Four properties, each with a failure mode a performance chart would hide.

| Property | What a bad value would mean |
| --- | --- |
| Holdings & trigger frequency | the rule fires so often that the strategy is a transaction-cost question, not an alpha one |
| Concentration | thirty names wearing a thirty-name label, or five names wearing one |
| Sector drift | the signal's structural bias — whichever sector it favours owns the book |
| Weight distribution | how lopsided the weighting really is |

Turnover is measured **target-to-target**. The realised figure is slightly lower, because between
rebalances the winners drift up on their own; that calculation needs drifted weights and belongs
to the backtest.

**This is where step 4, Portfolio Construction, lives today** — hand-rolled in the rule above. The
KaxaNuk Portfolio Construction library (MVO, HRP) is in development; when it lands, the weighting
half of section 2 becomes a library call and these diagnostics stay exactly as they are.

In [ ]:
turnover = (target_weights.diff().abs().sum(axis=1) / 2.0).rename("turnover_one_way")
turnover.iloc[0] = numpy.nan  # The first book is an initial build, not a rebalance.

effective_names = (1.0 / (target_weights ** 2).sum(axis=1)).rename("effective_names")
sorted_weights = numpy.sort(target_weights.to_numpy(), axis=1)[:, ::-1]
top5_share = pandas.Series(
    sorted_weights[:, :5].sum(axis=1), index=target_weights.index, name="top5_share",
)

portfolio_summary = pandas.concat(
    [
        (target_weights > 0).sum(axis=1).rename("holdings"),
        turnover,
        top5_share,
        effective_names,
        target_weights.max(axis=1).rename("max_weight"),
    ],
    axis=1,
)
portfolio_summary.index.name = "rebalance_date"

years = (REBALANCE_DATES[-1] - REBALANCE_DATES[0]).days / 365.25
print("Per-rebalance summary:")
print(portfolio_summary.describe().loc[["mean", "min", "max"]].round(3).to_string())
print(f"\n{len(REBALANCE_DATES):,} rebalances over {years:.1f} years"
      f" = {len(REBALANCE_DATES) / years:.0f} a year;"
      f" one-way turnover ~{turnover.sum() / years:.0%} a year")

# --- Sector exposure against the universe's own composition -----------------------------------
sector_by_company = company_metadata["sector"].fillna("(unclassified)")
sector_weights = (
    target_weights.T.groupby(sector_by_company.reindex(target_weights.columns).to_numpy())
    .sum().T
)
sector_weights.index.name = "rebalance_date"

universe_sector_share = (
    sector_by_company.reindex(target_weights.columns).value_counts(normalize=True)
    .reindex(sector_weights.columns).fillna(0.0)
)
average_sector_weight = sector_weights.mean()
sector_drift = pandas.DataFrame({
    "portfolio": average_sector_weight,
    "universe": universe_sector_share,
    "drift_pp": (average_sector_weight - universe_sector_share) * 100,
}).sort_values("drift_pp", ascending=False)

print("\nAverage sector weight vs the universe's own share (current classification):")
print(sector_drift.round(3).to_string())

In [ ]:
INK_SECONDARY = "#52514e"
SERIES_BLUE = "#2a78d6"
SERIES_ORANGE = "#eb6834"
SURFACE = "#fcfcfb"
GRID = "#ebeae5"

matplotlib.pyplot.rcParams.update({
    "figure.facecolor": SURFACE,
    "axes.facecolor": SURFACE,
    "axes.edgecolor": "#d8d7d2",
    "axes.labelcolor": INK_SECONDARY,
    "xtick.color": INK_SECONDARY,
    "ytick.color": INK_SECONDARY,
    "font.size": 10,
    "figure.dpi": 110,
    "savefig.dpi": 160,
    "savefig.bbox": "tight",
})


def style_axes(axes, title=None, subtitle=None, ylabel=None):
    """Left-aligned bold title over an optional subtitle line, with a recessive grid."""
    if title:
        axes.set_title(title, loc="left", pad=22 if subtitle else 8, weight="bold")
    if subtitle:
        axes.annotate(
            subtitle, xy=(0, 1), xycoords="axes fraction",
            xytext=(0, 6), textcoords="offset points",
            fontsize=9, color=INK_SECONDARY, va="bottom", ha="left",
        )
    if ylabel:
        axes.set_ylabel(ylabel)
    axes.grid(True, color=GRID, linewidth=0.8)
    axes.set_axisbelow(True)
    for side in ("top", "right"):
        axes.spines[side].set_visible(False)

    return axes


# --- The book over time ------------------------------------------------------------------------
panels = (
    (portfolio_summary["holdings"], "Holdings", "names", False),
    (portfolio_summary["turnover_one_way"], "One-way turnover per rebalance", "share", True),
    (portfolio_summary["top5_share"], "Top-5 concentration", "share of book", True),
    (portfolio_summary["effective_names"], "Effective number of names (1 / HHI)", "names", False),
)

figure, axes_grid = matplotlib.pyplot.subplots(2, 2, figsize=(13, 6.4), sharex=True)
for axes, (series, title, ylabel, is_share) in zip(axes_grid.flatten(), panels):
    axes.plot(series.index, series.to_numpy(), color=SERIES_BLUE, linewidth=1.1)
    if is_share:
        axes.yaxis.set_major_formatter(lambda value, _: f"{value:.0%}")
    style_axes(axes, title, ylabel=ylabel)
figure.suptitle(
    f"Experiment 1 book - top {HOLDING_COUNT}", x=0.01, ha="left", fontsize=13, weight="bold",
)
figure.tight_layout(rect=(0, 0, 1, 0.95))
figure.savefig(CHART_DIR / "portfolio_book_over_time.png")
matplotlib.pyplot.show()

# --- Sector drift --------------------------------------------------------------------------
ordered_drift = sector_drift.sort_values("drift_pp")
figure, axes = matplotlib.pyplot.subplots(figsize=(9, 4.6))
axes.barh(
    ordered_drift.index.astype(str),
    ordered_drift["drift_pp"].to_numpy(),
    color=[SERIES_BLUE if value >= 0 else SERIES_ORANGE for value in ordered_drift["drift_pp"]],
    height=0.72,
)
axes.axvline(0, color=INK_SECONDARY, linewidth=1.0)
axes.set_xlabel("portfolio weight minus universe share (percentage points)")
style_axes(axes, "What the benchmark is structurally tilted toward")
axes.spines["left"].set_visible(False)
figure.tight_layout()
figure.savefig(CHART_DIR / "portfolio_sector_drift.png")
matplotlib.pyplot.show()

# --- The weight distribution on the latest book ----------------------------------------------
latest = target_weights.iloc[-1]
latest = latest[latest > 0].sort_values(ascending=False)

figure, axes = matplotlib.pyplot.subplots(figsize=(11, 4.4))
axes.bar(range(len(latest)), latest.to_numpy(), color=SERIES_BLUE)
axes.set_xticks(range(len(latest)))
axes.set_xticklabels(
    [ticker_matrix.loc[target_weights.index[-1], company] for company in latest.index],
    rotation=90, fontsize=8,
)
axes.yaxis.set_major_formatter(lambda value, _: f"{value:.0%}")
style_axes(
    axes, f"The book on {target_weights.index[-1].date()}",
    subtitle=f"{len(latest)} names - largest {latest.iloc[0]:.1%}",
    ylabel="weight",
)
figure.tight_layout()
figure.savefig(CHART_DIR / "portfolio_latest_book.png")
matplotlib.pyplot.show()

print(f"Top 10 positions on {target_weights.index[-1].date()}:")
print(
    pandas.DataFrame({
        "ticker": ticker_matrix.loc[target_weights.index[-1], latest.index[:10]],
        "name": company_metadata.loc[latest.index[:10], "name"],
        "sector": company_metadata.loc[latest.index[:10], "sector"],
        "weight": latest[:10].map("{:.2%}".format),
    }).to_string(index=False)
)

### 3.1 · Write the deliverables

Two views of the same book, because two different readers need it:

- **`target_weights.csv`** — long, company-keyed, with names and sectors attached. The research
  view, and what a human reads.
- **`portfolio_weights.csv`** — wide, **ticker**-keyed, dates across the columns. The backtest
  engine's input: it looks up `<Ticker>.csv` in the market-data directory, so it has to speak in
  tickers, not in the ISIN-stitched companies. A company that changed ticker mid-history occupies
  two rows, each non-zero only while that listing was live — which is exactly right, because they
  are two files with two price histories.

In [ ]:
held_weights = target_weights.stack()
held_weights = held_weights[held_weights > 0]

target_weights_long = (
    pandas.concat(
        {
            "weight": held_weights,
            "ticker": ticker_matrix.loc[REBALANCE_DATES].stack().reindex(held_weights.index),
            "sizing": sizing_matrix.shift(1).loc[REBALANCE_DATES].stack()
                                     .reindex(held_weights.index),
        },
        axis=1,
    )
    .rename_axis(["rebalance_date", "company"])
    .reset_index()
    .join(company_metadata[["name", "sector", "industry"]], on="company")
    .sort_values(["rebalance_date", "weight"], ascending=[True, False])
)

# Shaped by the shared helper, so this file and the one section 4 hands the engine are built by
# one code path. The engine's weight file has no cash row -- every column must sum to 1.0 -- so
# any residual is parked in the cash proxy, keeping "go to cash" expressible rather than
# structurally impossible.
engine_weights = engine.to_engine_frame(target_weights, ticker_matrix, CASH_TICKER)
cash_weight = (
    engine_weights.loc[CASH_TICKER]
    if CASH_TICKER in engine_weights.index
    else pandas.Series(0.0, index=engine_weights.columns)
)

column_sums = engine_weights.sum(axis=0)
assert numpy.allclose(column_sums, 1.0, atol=1e-8), (
    f"engine weight columns must sum to 1.0; worst = {column_sums.min():.9f}"
)

missing_price_files = [
    ticker for ticker in engine_weights.index
    if not (MARKET_DATA_DIR / f"{ticker}.csv").is_file()
]
assert not missing_price_files, (
    f"no price file in {MARKET_DATA_DIR} for: {', '.join(missing_price_files[:10])}"
)

target_weights_long.to_csv(PORTFOLIO_DIR / "target_weights.csv", index=False)
portfolio_summary.to_csv(PORTFOLIO_DIR / "portfolio_summary.csv")
sector_weights.to_csv(PORTFOLIO_DIR / "sector_weights.csv")
engine_weights.to_csv(PORTFOLIO_DIR / "portfolio_weights.csv")

print(f"Written to {PORTFOLIO_DIR.relative_to(REPO_ROOT)}:")
print(f"  portfolio_weights.csv  {engine_weights.shape[0]:,} tickers x"
      f" {engine_weights.shape[1]:,} dates   <- backtest engine input")
print(f"  target_weights.csv     {len(target_weights_long):,} rows (rebalance x held name)")
print(f"  portfolio_summary.csv  {len(portfolio_summary):,} rows")
print(f"  sector_weights.csv     {sector_weights.shape[0]:,} x {sector_weights.shape[1]}")
print("  Charts/                3 PNGs")
print(f"\nCash weight: max {cash_weight.max():.4%}, mean {cash_weight.mean():.4%}")

---

## 4 · Backtest — KaxaNuk Backtest Engine

`portfolio_weights.csv` goes to the licensed engine, which simulates the book share by share: it
fills at a real price, charges commission per share on the unadjusted price, holds a cash reserve,
marks the portfolio daily between rebalances, and compares against the benchmarks.

**This is the only backtest in the repository.** There is deliberately no second, lighter
simulator: a simpler backtest that disagrees with the engine is worse than none at all, because it
lets the reader pick whichever number they prefer. Every figure quoted anywhere — here, in later
experiments, in `RESULTS.md` — comes from this engine.

Benchmarks are **SPY**, **QQQ** and **KN600** (the KaxaNuk US equity index). The window is clipped
to the shortest benchmark up front rather than discovered as a crash: the engine validates that
every benchmark spans the whole period and raises otherwise.

> **Environment note.** `kaxanuk-backtest-engine` installs from KaxaNuk's licensed index rather
> than PyPI (see `README.md`). The cell guards the import so the notebook stays runnable without a
> licence — it reports and skips rather than raising, and every portfolio deliverable above is
> produced either way.

In [ ]:
import importlib.util
import logging
import os

import dotenv

dotenv.load_dotenv(REPO_ROOT / "Config" / ".env")

BENCHMARK_TICKERS = engine.BENCHMARK_TICKERS
ENGINE_AVAILABLE = importlib.util.find_spec("kaxanuk.backtest_engine") is not None

if not ENGINE_AVAILABLE:
    print("kaxanuk-backtest-engine is not installed - skipping the backtest.")
    print("  Install it from the licensed index (see README.md), then re-run this cell.")
    print("\nEverything in Portfolio/ is already written and does not depend on the engine.")
    benchmark_run = None
else:
    assert os.getenv("KNBE_API_KEY_KAXANUK"), (
        "KNBE_API_KEY_KAXANUK not in the environment; add it to Config/.env"
    )

    # The engine validates that every benchmark spans the whole window, so the end is clipped to
    # the shortest series up front rather than discovered as a BenchmarkAlignmentError.
    # A clone without the hand-supplied KN600.csv measures against SPY and QQQ instead of
    # failing; the helper says which ones it dropped and why.
    benchmark_tickers = engine.available_benchmarks(MARKET_DATA_DIR, BENCHMARK_TICKERS)
    benchmark_last_dates = {
        ticker: pandas.read_csv(
            MARKET_DATA_DIR / f"{ticker}.csv", usecols=[DATE_COLUMN], parse_dates=[DATE_COLUMN],
        )[DATE_COLUMN].max()
        for ticker in benchmark_tickers
    }
    backtest_end = min(REBALANCE_DATES[-1], *benchmark_last_dates.values())

    print("Benchmark coverage ends:")
    for ticker, last_date in benchmark_last_dates.items():
        marker = "  <- binding" if last_date == backtest_end else ""
        print(f"  {ticker:<6} {last_date.date()}{marker}")
    print(f"\nBacktest window: {REBALANCE_DATES[0].date()} -> {backtest_end.date()}")

    engine_settings = engine.EngineSettings(
        market_data_directory=MARKET_DATA_DIR,
        portfolio_directory=PORTFOLIO_DIR,
        output_directory=BACKTEST_DIR,
        start_date=REBALANCE_DATES[0].date(),
        end_date=backtest_end.date(),
        benchmark_tickers=benchmark_tickers,
    )
    # `engine_weights` is the very file written in section 3.1 -- the engine reads that CSV.
    benchmark_run = engine.run_variant(
        "portfolio_weights",
        engine_weights,
        engine_settings,
    )

    print(f"\nResults: {benchmark_run.workbook_path.name}\n")
    print(benchmark_run.summary.to_string())

### 4.1 · The record, drawn from the engine's own daily series

Both charts read `portfolio_bench_total_value` out of the workbook above, so what is plotted is
exactly what the engine simulated — not a re-derivation of it.

In [ ]:
if benchmark_run is None:
    print("No engine run - nothing to chart.")
else:
    values = benchmark_run.daily_values
    equity = values / values.iloc[0]
    equity.columns = [column.replace("_Value", "") for column in equity.columns]
    # The engine reports both the invested book and the book including its cash reserve; the
    # second is the one that corresponds to the capital actually committed.
    equity = equity.drop(columns=["Portfolio"], errors="ignore")
    equity = equity.rename(columns={"Total_Portfolio": "Experiment 1"})
    drawdown = equity / equity.cummax() - 1.0

    figure, axes_grid = matplotlib.pyplot.subplots(
        2, 1, figsize=(12, 7.4), sharex=True, gridspec_kw={"height_ratios": [2, 1]},
    )
    palette = (SERIES_BLUE, INK_SECONDARY, "#9db4cc", "#2f9e6b")
    for column, color in zip(equity.columns, palette):
        axes_grid[0].plot(equity.index, equity[column], linewidth=1.6, color=color, label=column)
        axes_grid[1].plot(drawdown.index, drawdown[column], linewidth=1.1, color=color)

    axes_grid[0].set_yscale("log")
    axes_grid[0].legend(frameon=False, fontsize=9)
    style_axes(
        axes_grid[0], "Growth of 1.00, net of commission",
        subtitle="log scale; KaxaNuk Backtest Engine daily marks",
        ylabel="multiple of capital",
    )
    axes_grid[1].yaxis.set_major_formatter(lambda value, _: f"{value:.0%}")
    style_axes(axes_grid[1], "Drawdown", ylabel="from peak")
    figure.tight_layout()
    figure.savefig(CHART_DIR / "backtest_engine.png")
    matplotlib.pyplot.show()

    annual = equity.resample("YE").last().pct_change()
    annual.iloc[0] = equity.resample("YE").last().iloc[0] - 1.0
    annual.index = annual.index.year
    print("Calendar-year returns:")
    print(annual.map(lambda value: f"{value:.1%}" if pandas.notna(value) else "-").to_string())

---

## 5 · Attribution — KaxaNuk Attribution Analysis

The backtest says *how much* the book made; attribution says **where it came from**:

- **Brinson-Fachler** splits active return into an **allocation** effect (being overweight the
  right sectors) and a **selection** effect (picking the right names inside them).
- **KN5FM** regresses the book against KaxaNuk's factor returns (market, size, value, momentum,
  residual volatility, beta, plus per-sector factors) and leaves a residual — the part the
  factors cannot explain.

This stage exists to answer one blunt question: **is this book the signal, or a factor exposure
wearing the signal's name?** Expect the answer to be partial. An *absolute* rule — a name judged
against its own history — is close to invisible to a factor model built on *relative* factors, so
a book can beat every benchmark while the model assigns ~0% to the factor its thesis is named
after. That is a finding, not a failure; the follow-up is the counterfactual book with the signal
switched off, and the selection / sizing / timing decomposition by counterfactual books.

### The two benchmark inputs, built from the supplied index data

The library wants the benchmark as **weights** plus a **daily return series**, both in the
horizontal `Ticker × dates` layout its loader auto-detects, with no nulls. Neither exists in that
shape, so both are derived here from the index files in `Data/Curator/Benchmarks/`:

| Library input | Built from | Note |
| --- | --- | --- |
| `benchmark_weights.csv` | the index holdings file | daily constituent weights; rows already sum to 1.0 |
| `benchmark_returns.csv` | the index returns file | transposed to a single row, named and date-parsed per the declaration |

Both source file names are constants in **`Experiments/engine.py`**, alongside the benchmark
tickers every experiment is scored against. Switching to a different index is an edit there —
no code in this notebook names a file.

**What binds the window.** `start_date="auto"` intersects the factor files and the benchmark
holdings, so the attribution describes a *shorter* period than the backtest above. They are not
directly comparable, and that is a property of the inputs, not a bug. Record both windows in
`FINDINGS_1.md`.

Two honest caveats to carry into any reading of the numbers:

1. The library prices only the tickers in *our* portfolio, so benchmark constituents we do not
   hold arrive without returns. The allocation/selection split is indicative, not exact.
2. Sector buckets come from `sector_current`, which is **today's** classification. Every period
   before a reclassification is misattributed — see the warning in `Data/refinery.py`.

In [ ]:
ATTRIBUTION_AVAILABLE = importlib.util.find_spec("kaxanuk.attribution_analysis") is not None

# All attribution inputs live inside this repo rather than in an external checkout.  The file
# names and the index's date convention are constants in Experiments/engine.py, next to the
# benchmark tickers, so pointing this stage at a different index is one edit in one file.
FACTOR_MODELS_DIR = REPO_ROOT / "Data" / "Curator" / "Factors"
INDEX_HOLDINGS_PATH = BENCHMARK_DIR / engine.ATTRIBUTION_HOLDINGS_FILE
INDEX_RETURNS_PATH = BENCHMARK_DIR / engine.ATTRIBUTION_RETURNS_FILE
BENCHMARK_WEIGHTS_NAME = "benchmark_weights"
BENCHMARK_RETURNS_NAME = "benchmark_returns"
ATTRIBUTION_DASHBOARD_PORT = 8051


# Everything this library produces is a side effect: log lines and matplotlib figures.
# Under the notebook's inline backend `matplotlib.pyplot.show()` renders a figure and then
# *closes* it, so there is nothing left to save afterwards. Intercepting `show` is what lets
# the figures be both displayed inline and written to disk for FINDINGS_1.md to cite.
ATTRIBUTION_FIGURE_NAMES = (
    "attribution_brinson_fachler.png",
    "attribution_factor_model.png",
)
ATTRIBUTION_FIGURE_PATHS = []
ORIGINAL_PYPLOT_SHOW = matplotlib.pyplot.show


def save_then_show(*arguments, **keyword_arguments):
    """Write every currently open figure into the attribution folder, then show it as usual."""
    for number in matplotlib.pyplot.get_fignums():
        position = len(ATTRIBUTION_FIGURE_PATHS)
        name = (
            ATTRIBUTION_FIGURE_NAMES[position]
            if position < len(ATTRIBUTION_FIGURE_NAMES)
            else f"attribution_figure_{position}.png"
        )
        path = ATTRIBUTION_DIR / name
        matplotlib.pyplot.figure(number).savefig(path, dpi=160, bbox_inches="tight")
        ATTRIBUTION_FIGURE_PATHS.append(path)

    return ORIGINAL_PYPLOT_SHOW(*arguments, **keyword_arguments)


def write_horizontal(frame, destination):
    """
    Write `frame` as the Ticker x ISO-date table both KaxaNuk loaders auto-detect.

    Defined once because it is a contract with the library rather than a formatting choice: get
    the index name or the date format wrong and the loader mis-detects the orientation instead of
    failing outright.
    """
    output = frame.copy()
    output.columns = pandas.DatetimeIndex(output.columns).strftime("%Y-%m-%d")
    output.index.name = "Ticker"
    output.to_csv(destination)

    return output


def build_benchmark_weights(source, destination):
    """The index's daily holdings, transposed to Ticker x dates. Nulls are not allowed, so 0.0."""
    holdings = pandas.read_csv(source, parse_dates=["m_date"]).set_index("m_date")

    return write_horizontal(holdings.fillna(0.0).T, destination)


def build_benchmark_returns(source, destination):
    """The index's daily returns as a single-row Ticker x dates table."""
    # Whether the dates read day-first is declared in engine.py: pandas cannot infer it, and
    # guessing wrong silently shifts the entire series.
    returns = pandas.read_csv(
        source,
        parse_dates=["m_date"],
        dayfirst=engine.ATTRIBUTION_RETURNS_DAY_FIRST,
    ).set_index("m_date")
    series = returns.iloc[:, 0].rename(engine.ATTRIBUTION_RETURNS_SERIES_NAME).dropna()

    return write_horizontal(series.to_frame().T, destination)


if not ATTRIBUTION_AVAILABLE:
    print("kaxanuk-attribution-analysis is not installed - skipping attribution.")
    print("  Same licensed-index install as the backtest engine.")
elif not FACTOR_MODELS_DIR.is_dir() or not any(FACTOR_MODELS_DIR.glob(engine.FACTOR_FILE_PATTERN)):
    print(f"No factor files in {FACTOR_MODELS_DIR} - skipping attribution.")
else:
    benchmark_weights_frame = build_benchmark_weights(
        INDEX_HOLDINGS_PATH, BENCHMARK_DIR / f"{BENCHMARK_WEIGHTS_NAME}.csv",
    )
    benchmark_returns_frame = build_benchmark_returns(
        INDEX_RETURNS_PATH, BENCHMARK_DIR / f"{BENCHMARK_RETURNS_NAME}.csv",
    )
    print(f"benchmark_weights.csv : {benchmark_weights_frame.shape[0]:,} tickers x"
          f" {benchmark_weights_frame.shape[1]:,} dates")
    print(f"benchmark_returns.csv : 1 x {benchmark_returns_frame.shape[1]:,} dates")

    factor_files = sorted(FACTOR_MODELS_DIR.glob(engine.FACTOR_FILE_PATTERN))
    print(f"factor files          : {len(factor_files)}  <- record this count in FINDINGS_1.md")

    unpriced = set(benchmark_weights_frame.index) - set(engine_weights.index)
    print(f"\nBenchmark names the library cannot price: {len(unpriced):,} of"
          f" {benchmark_weights_frame.shape[0]:,}"
          f" ({len(unpriced) / benchmark_weights_frame.shape[0]:.0%})"
          " - it loads prices only for tickers in our own book,")
    print("so read Brinson-Fachler as indicative rather than exact.\n")
    from kaxanuk.attribution_analysis.entities.configuration import (
        Configuration as AttributionConfiguration,
    )
    from kaxanuk.attribution_analysis.performance_attribution import main as attribution_main

    assert os.getenv("KNAA_API_KEY_KAXANUK"), (
        "KNAA_API_KEY_KAXANUK not in the environment; add it to Config/.env"
    )

    attribution_configuration = AttributionConfiguration(
        input_directory=str(EXPERIMENT_DIR),
        weights_portfolio_directory=str(PORTFOLIO_DIR),
        weights_benchmark_directory=str(BENCHMARK_DIR),
        benchmark_returns_directory=str(BENCHMARK_DIR),
        investable_assets_directory=str(MARKET_DATA_DIR),
        factor_returns_by_factor_directory=str(FACTOR_MODELS_DIR),
        portfolio_file_name="portfolio_weights",
        benchmark_file_name=BENCHMARK_WEIGHTS_NAME,
        benchmark_return_file_name=BENCHMARK_RETURNS_NAME,
        user_column_date=DATE_COLUMN,
        user_column_price=MARK_PRICE_COLUMN,
        market_data_input_format="csv",
        portfolio_input_format="csv",
        start_date="auto",
        end_date="auto",
        brinson_fachler_method=True,
        factor_model_method=True,
    )

    # The library reports through the logging module and returns None, so a handler on the root
    # logger is what makes its numbers visible here at all.
    logging.basicConfig(level=logging.INFO, format="%(message)s", stream=sys.stdout, force=True)
    logging.getLogger("kaxanuk.attribution_analysis").setLevel(logging.INFO)
    matplotlib.pyplot.show = save_then_show

    attribution_main(
        attribution_configuration,
        launch_dashboard=False,
        dashboard_port=ATTRIBUTION_DASHBOARD_PORT,
    )
    matplotlib.pyplot.show = ORIGINAL_PYPLOT_SHOW

    logging.getLogger().handlers.clear()  # Leave the notebook's logging as we found it.
    print(f"\nAttribution figures written to {ATTRIBUTION_DIR.relative_to(REPO_ROOT)}")
    for path in sorted(ATTRIBUTION_DIR.iterdir()):
        if path.name != ".gitkeep":
            print(f"  {path.name}  ({path.stat().st_size / 1024:,.0f} KB)")

---

## 6 · Verdict

<What this notebook concluded, in words. A notebook that ends in a number and no sentence gets read
as whatever the reader hoped. Three sentences: does the book work; what attribution says about why;
what the next experiment should change. Then copy the numbers into `FINDINGS_1.md` — that file is
the record, this notebook is the method.>

---

## Handoff

| Output | Consumed by |
| --- | --- |
| `Portfolio/portfolio_weights.csv` | the backtest engine and the attribution library |
| `Portfolio/target_weights.csv` | humans, and `FINDINGS_1.md` |
| `Portfolio/portfolio_summary.csv` | `FINDINGS_1.md` |
| `Portfolio/Charts/*.png` | `FINDINGS_1.md` |
| `Backtest/`, `Attribution/` | `FINDINGS_1.md`, and the comparison baseline for every later experiment |

Later experiments read the **same** panel, over the same window, with the same costs and the same
rebalancing convention, and change only the selection or the weighting — which is what makes the
comparison against this benchmark meaningful.

In [ ]:
summary = pandas.DataFrame({
    "metric": [
        "companies in the panel",
        "trading days",
        "book starts",
        "book ends",
        "holdings",
        "rebalances",
        "rebalances per year",
        "one-way turnover per year",
        "top-5 weight share (mean)",
        "effective number of names (mean)",
        "largest single weight",
        "backtest engine available",
    ],
    "value": [
        f"{signal_matrix.shape[1]}",
        f"{len(TRADING_DAYS):,}",
        f"{REBALANCE_DATES[0].date()}",
        f"{REBALANCE_DATES[-1].date()}",
        f"{HOLDING_COUNT}",
        f"{len(REBALANCE_DATES):,}",
        f"{len(REBALANCE_DATES) / years:.0f}",
        f"{turnover.sum() / years:.0%}",
        f"{portfolio_summary['top5_share'].mean():.1%}",
        f"{portfolio_summary['effective_names'].mean():.1f}",
        f"{target_weights.max().max():.1%}",
        "yes" if ENGINE_AVAILABLE else "no - see the note in section 4",
    ],
})
print(summary.to_string(index=False))

---

## Open items

| # | Item | Why it matters |
| --- | --- | --- |
| 1 | **The attribution window is shorter than the backtest**, bound by factor-file coverage. | The two sets of numbers describe different periods and must not be compared directly. |
| 2 | **Delisting exits use one day of hindsight.** A position is sold on the last day it still has a fill price, which is only knowable the day after. | The standard backtest compromise, stated in section 2; it flatters the strategy only to the extent a real desk would have exited a day later at a worse price. |
| 3 | **Dual share classes are two positions.** Names with two ISINs cannot be merged by the company key. | The economic bet on one issuer is doubled whenever both classes are held. |
| 4 | <The benchmark's own known weakness — churn at a rank cut-off, a slow exit — measured here and deliberately left for a later experiment.> | |